# Face2Art on Colab
Use a GPU runtime. Training outputs and resumable checkpoints are stored in Google Drive.

In [ ]:
%cd /content
!test -d /content/face2art/.git || git clone --depth 1 https://github.com/karuniaperjuangan/face2art.git /content/face2art
%cd /content/face2art
!git pull --ff-only
%pip install -q -U "gdown>=6,<7" huggingface-hub onnx onnx2torch pillow pyyaml requests "wandb>=0.25,<1" "gradio>=6,<7"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%run -m src.download -- --dataset ffhq --dataset sngfaces

In [ ]:
from pathlib import Path
from getpass import getpass
import os
import yaml

USE_WANDB = False # @param {type:"boolean"}
WANDB_PROJECT = "face2art" # @param {type:"string"}
WANDB_ENTITY = "" # @param {type:"string"}
WANDB_API_KEY = "" # @param {type:"string"}
drive_root = Path('/content/drive/MyDrive/face2art')
drive_root.mkdir(parents=True, exist_ok=True)
config = yaml.safe_load(Path('config/train.yaml').read_text())
config['experiment']['output_dir'] = str(drive_root / 'outputs')
config['training']['device'] = 'cuda'
if USE_WANDB:
    os.environ['WANDB_API_KEY'] = WANDB_API_KEY
    config['wandb'].update(enabled=True, project=WANDB_PROJECT, entity=WANDB_ENTITY or None)
latest = drive_root / 'outputs' / config['experiment']['name'] / 'checkpoints/latest.pt'
config['training']['resume'] = str(latest) if latest.exists() else None
CONFIG_PATH = str(drive_root / 'train_colab.yaml')
Path(CONFIG_PATH).write_text(yaml.safe_dump(config, sort_keys=False))
print('Outputs:', config['experiment']['output_dir'])
print('Resume:', config['training']['resume'])

In [ ]:
%run -m src.train -- --config "$CONFIG_PATH"

In [ ]:
run_dir = drive_root / 'outputs' / config['experiment']['name'] / 'checkpoints'
checkpoint = run_dir / 'best.pt'
if not checkpoint.exists():
    checkpoint = run_dir / 'latest.pt'
assert checkpoint.exists(), f'No checkpoint found in {run_dir}'
CHECKPOINT_PATH = str(checkpoint)
%run -m src.ui -- --config "$CONFIG_PATH" --checkpoint "$CHECKPOINT_PATH" --share